In [1]:
import ast
import json
from glob import glob

import pandas as pd

In [2]:
BASE = "/Users/charles-/Downloads/Master/M2/PersonasData/Stupeflip"
path_to_songs = f"{BASE}/songsWithRhymeSchemes/*.json"

In [3]:
def add_persona_id(sections, persona_id="kingju"):
    for section in sections:
        for line in section.get("lines", []):
            line["persona"] = persona_id
    return sections


def load_songs(path_pattern, ids, persona_id):
    """Charge les JSON, garde les ids voulus et ajoute la persona à chaque ligne."""
    rows = []
    for path in glob(path_pattern):
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        if song["song_id"] in ids:
            song["lyrics"] = add_persona_id(song["lyrics"], persona_id)
            rows.append(song)
    return pd.DataFrame(rows)

In [4]:
ids_kingju = set(pd.read_csv(f"{BASE}/stupeflip_kingju.csv")["id"])
df_kingju = load_songs(path_to_songs, ids_kingju, "kingju")

In [5]:
# pophip.csv est encore au format repr Python : on le lit une dernière fois avec ast
df_pophip = pd.read_csv(f"{BASE}/pophip.csv")
df_pophip["lyrics"] = df_pophip["lyrics"].apply(ast.literal_eval)

In [6]:
df_stupeflip = pd.concat([df_pophip, df_kingju], ignore_index=True)

In [7]:
# Sauvegarde : lyrics sérialisé en vrai JSON
df_stupeflip.assign(lyrics=df_stupeflip["lyrics"].apply(json.dumps, ensure_ascii=False)) \
    .to_csv(f"{BASE}/stupeflip.csv", index=False)

In [8]:
def save_df(df, path):
    """Sauvegarde en CSV avec lyrics sérialisé en vrai JSON."""
    df.assign(lyrics=df["lyrics"].apply(json.dumps, ensure_ascii=False)).to_csv(path, index=False)

In [9]:
save_df(df_pophip, f"{BASE}/pophip.csv")
save_df(df_kingju, f"{BASE}/kingju.csv")
save_df(df_stupeflip, f"{BASE}/stupeflip.csv")